# Azzaro: Whisper turbo vs small vs large

Comparamos tres modelos de Whisper sobre el mismo video, contamos diferencias de palabras y revisamos los clips donde no coinciden. El caso puntual a mirar es `racingista` / `racinguista`.

## 1. Configuracion

In [ ]:
from itertools import combinations
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Video, display

from evaluation.src.whisper_model_comparison import (
    alinear_diferencias,
    descargar_video_yt,
    exportar_clips_diferencias,
    extraer_palabras,
    resumen_diferencias,
    transcribir_whisper,
)

VIDEO_URL = "https://www.youtube.com/watch?v=a4ggqJZXnQE"
VIDEO_PATH = None  # si ya tenes el mp4 local, poner aca la ruta y dejar VIDEO_URL como esta

MODELOS = ["turbo", "small", "large"]
COOKIES_FROM_BROWSER = "chrome"  # poner None si YouTube no pide login
FORCE_TRANSCRIBE = False

MAX_DIFFERENCE_CLIPS = 80
MARGIN_SECONDS = 1.25
PATRON_REVISION = r"racingista|racinguista|racing"

WORKDIR = ROOT / "evaluation" / "outputs" / "azzaro_whisper"
WORKDIR

## 2. Descargar o cargar video

In [ ]:
if VIDEO_PATH:
    video_path = Path(VIDEO_PATH).expanduser().resolve()
else:
    video_path = descargar_video_yt(
        VIDEO_URL,
        WORKDIR / "videos",
        nombre_base="azzaro_racing_caracas",
        cookies_from_browser=COOKIES_FROM_BROWSER,
    )

video_path

## 3. Transcribir con turbo, small y large

Esto cachea cada JSON en `evaluation/outputs/azzaro_whisper/transcripts/`. Si ya existe, no vuelve a transcribir.

In [ ]:
resultados = {}
palabras = {}

for model_name in MODELOS:
    print(f"Transcribiendo / cargando cache: {model_name}")
    resultados[model_name] = transcribir_whisper(
        video_path,
        model_name=model_name,
        output_dir=WORKDIR / "transcripts",
        force=FORCE_TRANSCRIBE,
    )
    palabras[model_name] = extraer_palabras(resultados[model_name], model_name)

pd.DataFrame([
    {"modelo": modelo, "palabras": len(palabras_modelo)}
    for modelo, palabras_modelo in palabras.items()
])

## 4. Alinear palabras y contar diferencias

Se comparan los tres pares: `turbo-small`, `turbo-large` y `small-large`. Despues se exportan clips cortos para revisar manualmente.

In [ ]:
todas_las_diferencias = []
resumenes = []

for modelo_a, modelo_b in combinations(MODELOS, 2):
    diferencias = alinear_diferencias(
        palabras[modelo_a],
        palabras[modelo_b],
        contexto=5,
    )
    resumen = resumen_diferencias(diferencias, palabras[modelo_a], palabras[modelo_b])
    resumen["comparacion"] = f"{modelo_a}_vs_{modelo_b}"
    resumenes.append(resumen)

    for diferencia in diferencias:
        diferencia = dict(diferencia)
        diferencia["comparacion"] = f"{modelo_a}_vs_{modelo_b}"
        diferencia["modelo_a"] = modelo_a
        diferencia["modelo_b"] = modelo_b
        diferencia["texto_a"] = diferencia["turbo_text"]
        diferencia["texto_b"] = diferencia["small_text"]
        diferencia["contexto_a"] = diferencia["contexto_turbo"]
        diferencia["contexto_b"] = diferencia["contexto_small"]
        todas_las_diferencias.append(diferencia)

for diff_id, diferencia in enumerate(todas_las_diferencias, start=1):
    diferencia["diff_id"] = diff_id

df_resumen = pd.DataFrame(resumenes)
display(df_resumen)

diferencias_con_clips = exportar_clips_diferencias(
    video_path,
    todas_las_diferencias,
    output_dir=WORKDIR / "clips_diferencias",
    margen=MARGIN_SECONDS,
    max_clips=MAX_DIFFERENCE_CLIPS,
)

df_clips = pd.DataFrame(diferencias_con_clips)
if "review_index" not in df_clips.columns:
    df_clips.insert(0, "review_index", range(len(df_clips)))

df_resumen.to_csv(WORKDIR / "resumen_turbo_small_large.csv", index=False)
df_clips.to_csv(WORKDIR / "diferencias_con_clips.csv", index=False)

cols = ["review_index", "comparacion", "diff_id", "start", "end", "texto_a", "texto_b", "clip_path"]
display(df_clips[cols])

## 5. Ver diferencias una por una

Cambiar `REVIEW_INDEX` para avanzar: `0`, `1`, `2`, etc. Despues de mirar el clip, completar `MODELO_CORRECTO` y correr la celda de guardado.

In [ ]:
# Opcional: primero miramos rapido los casos donde aparece racing/racingista/racinguista.
mask_racing = (
    df_clips["texto_a"].str.contains(PATRON_REVISION, case=False, na=False, regex=True)
    | df_clips["texto_b"].str.contains(PATRON_REVISION, case=False, na=False, regex=True)
    | df_clips["contexto_a"].str.contains(PATRON_REVISION, case=False, na=False, regex=True)
    | df_clips["contexto_b"].str.contains(PATRON_REVISION, case=False, na=False, regex=True)
)

display(df_clips.loc[mask_racing, ["review_index", "comparacion", "diff_id", "texto_a", "texto_b", "contexto_a", "contexto_b", "clip_path"]])

In [ ]:
REVIEW_INDEX = 0

row = df_clips.loc[df_clips["review_index"].eq(REVIEW_INDEX)].iloc[0]

print(f"revision {REVIEW_INDEX + 1} de {len(df_clips)}")
print("comparacion:", row["comparacion"])
print("diff_id:", row["diff_id"])
print("tiempo:", row["start"], "-", row["end"])
print("-" * 80)
print(row["modelo_a"], ":", row["texto_a"])
print(row["modelo_b"], ":", row["texto_b"])
print("-" * 80)
print("contexto", row["modelo_a"], ":", row["contexto_a"])
print("contexto", row["modelo_b"], ":", row["contexto_b"])

display(Video(row["clip_path"], embed=True, html_attributes="controls"))

In [ ]:
MODELO_CORRECTO = ""  # opciones: "turbo", "small", "large", "ninguno", "empate"
NOTA = ""

revision_path = WORKDIR / "revision_manual_turbo_small_large.csv"

if revision_path.exists():
    revision = pd.read_csv(revision_path)
else:
    revision = df_clips.copy()
    revision["modelo_correcto"] = ""
    revision["nota"] = ""

if MODELO_CORRECTO:
    mask = revision["review_index"].eq(REVIEW_INDEX)
    revision.loc[mask, "modelo_correcto"] = MODELO_CORRECTO
    revision.loc[mask, "nota"] = NOTA
    revision.to_csv(revision_path, index=False)
    display(revision.loc[mask, ["review_index", "comparacion", "texto_a", "texto_b", "modelo_correcto", "nota"]])
    print(f"Guardado en: {revision_path}")
else:
    completadas = revision["modelo_correcto"].fillna("").ne("").sum()
    print(f"Completa MODELO_CORRECTO para guardar. Revisadas: {completadas} / {len(revision)}")
    display(revision[["review_index", "comparacion", "texto_a", "texto_b", "modelo_correcto", "nota"]])